Test model

In [ ]:
!pip install -U transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import gc

model_name = "Qwen/Qwen2.5-3B-Instruct"

# เช็คว่ามีโมเดลอยู่หรือยัง ถ้าไม่มีค่อยโหลด
if 'model' not in globals():
    print("🚀 กำลังโหลดโมเดลเข้า GPU...")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto"
    )
    
    # สร้างประวัติการคุยเริ่มต้น
    chat_history = [{"role": "system", "content": "You are a helpful assistant. Please respond in Thai."}]
    print("✅ โหลดสำเร็จ! พร้อมใช้งาน")
else:
    print("📢 โมเดลอยู่ในแรมแล้ว ข้ามการโหลดเพื่อประหยัด VRAM")

In [ ]:
# --- แก้คำถามตรงนี้ ---
user_input = "ช่วยอธิบายวิธีทำต้มยำกุ้งแบบสั้นๆ หน่อย"

# 1. เคลียร์แรมเก่าก่อนเริ่มรอบใหม่
gc.collect()
torch.cuda.empty_cache()

# 2. จำกัดความจำ (ถ้าคุยยาวเกิน 8 ประโยค ให้ลบอันเก่าทิ้งเพื่อไม่ให้ VRAM เต็ม)
if len(chat_history) > 8:
    chat_history = [chat_history[0]] + chat_history[-6:]

# 3. ใส่คำถามใหม่ลงไป
chat_history.append({"role": "user", "content": user_input})

# 4. เตรียมข้อมูลและ Generate
text = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

# 5. แปลผลลัพธ์
generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# 6. บันทึกคำตอบและแสดงผล
chat_history.append({"role": "assistant", "content": response})

print(f"👤 คุณ: {user_input}")
print(f"🤖 AI: {response}")

### จดจำ Chat เดิม

In [ ]:
# --- ใส่คำถามของคุณตรงนี้ ---
user_input = "แล้วกุ้งแม่น้ำล่ะ ทำต้มยำเหมือนกันไหม?"

# 1. เพิ่มคำถามใหม่ลงในประวัติ
chat_history.append({"role": "user", "content": user_input})

# 2. เตรียม Prompt จากประวัติทั้งหมด
text = tokenizer.apply_chat_template(
    chat_history,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# 3. ให้ AI สร้างคำตอบ
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7
)

# 4. ตัดส่วนคำถามออกให้เหลือแค่คำตอบใหม่
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# 5. บันทึกคำตอบของ AI ลงในประวัติ เพื่อใช้คุยต่อในรอบหน้า
chat_history.append({"role": "assistant", "content": response})

# --- แสดงผล ---
print(f"👤 คุณ: {user_input}")
print(f"🤖 AI: {response}")
print(f"\n(ประวัติการคุยตอนนี้มีทั้งหมด {len(chat_history)} ข้อความ)")

ถ้าอยากเริ่มคุยเรื่องใหม่ (ล้างสมอง): ให้กลับไปรัน Cell 2 อีกรอบ (หรือเพิ่มบรรทัด chat_history = [{"role": "system", "content": "..."}] ใน Cell 3 แล้วรันหนึ่งครั้ง) เพื่อล้างประวัติเก่าครับ

ถ้า AI ตอบสั้นไป: ปรับค่า max_new_tokens ใน Cell 3 ให้มากขึ้น (เช่น 1024)

ถ้าอยากให้ AI มีบุคลิกเปลี่ยนไป: แก้ไขข้อความใน system prompt ที่อยู่ใน Cell 2 ครับ